# 03: Simple Prediction - Linear Regression by Intuition

## Your First ML Model!

We're going to build something that **learns from data** and **makes predictions**. This is the essence of Machine Learning!

We'll start with the simplest model: **Linear Regression** - finding the best line through data.

### The Web Dev Analogy

Think of this like fitting a trend line in analytics:
- You have data points (page views vs conversions)
- You want to find the relationship (line)
- So you can predict new values

The magic: **the computer finds the best line automatically!**

## What You'll Learn
- [ ] Implement linear regression using gradient descent from scratch
- [ ] Explain how the MSE loss function measures prediction error
- [ ] Compare the effect of different learning rates on training convergence

## Connection to Previous Lessons

| What you learned | How it connects here |
|-----------------|---------------------|
| **Lesson 1**: NumPy operations (`@`, `np.sum`, etc.) | These power all the math behind gradient descent |
| **Lesson 2**: Features and labels | Features (house size) and labels (price) become our training data |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print("Ready to make predictions! 🎯")

## 1. The Problem: Fitting a Line

Let's say we have data about house sizes and prices. We want to predict price from size.

In [ ]:
# Generate some fake house data
n_houses = 50

# Size in hundreds of sq ft (10-30 = 1000-3000 sq ft)
sizes = np.random.uniform(10, 30, n_houses)

# Price = base + size_effect + noise
# True relationship: price = 50 + 10 * size (with noise)
true_slope = 10
true_intercept = 50
noise = np.random.normal(0, 20, n_houses)
prices = true_intercept + true_slope * sizes + noise

# Plot the data
plt.figure(figsize=(10, 6))
plt.scatter(sizes, prices, alpha=0.7, s=60, edgecolors='black', linewidth=0.5)
plt.xlabel('Size (hundreds of sq ft)')
plt.ylabel('Price (thousands of $)')
plt.title('House Sizes vs Prices')
plt.grid(True, alpha=0.3)
plt.show()

print("\n🤔 Can you see the trend? There's a relationship here...")

## 2. The Line Equation

A line is defined by:

$$y = mx + b$$

Or in ML terms:

$$\text{prediction} = \text{weight} \times \text{feature} + \text{bias}$$

We need to find the **best** values for weight (slope) and bias (intercept).

In [ ]:
# Let's try different lines and see how well they fit
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

lines = [
    (5, 100, 'Too Flat'),
    (10, 50, 'Good Fit!'),
    (20, 0, 'Too Steep'),
]

x_line = np.array([8, 32])

for idx, (slope, intercept, title) in enumerate(lines):
    ax = axes[idx]
    
    # Data points
    ax.scatter(sizes, prices, alpha=0.6, s=40)
    
    # Line
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, 'r-', linewidth=2, label=f'y = {slope}x + {intercept}')
    
    ax.set_xlabel('Size')
    ax.set_ylabel('Price')
    ax.set_title(title)
    ax.legend()
    ax.set_xlim(8, 32)
    ax.set_ylim(50, 400)

plt.tight_layout()
plt.show()

print("\n🎯 The middle line looks best! But how do we know for sure?")

## 3. The Loss Function: "How Wrong Am I?"

We need a way to measure how bad a line is. This is called the **loss function** (or cost function).

The most common loss for regression: **Mean Squared Error (MSE)**

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (\text{prediction}_i - \text{actual}_i)^2$$

In plain English: "Average of squared mistakes"

Why squared?
- Makes all errors positive (can't cancel out)
- Penalizes big errors more than small ones

In [ ]:
def predict(X, weight, bias):
    """Make predictions: y = weight * x + bias"""
    return weight * X + bias

def mse_loss(predictions, actuals):
    """Calculate Mean Squared Error."""
    errors = predictions - actuals
    squared_errors = errors ** 2
    return np.mean(squared_errors)

# Calculate loss for each line
print("Loss (MSE) for each line:")
print("=" * 40)

for slope, intercept, name in lines:
    preds = predict(sizes, slope, intercept)
    loss = mse_loss(preds, prices)
    print(f"{name:15} (y = {slope}x + {intercept}): MSE = {loss:.2f}")

print("\n✅ Lower MSE = Better fit!")

In [ ]:
# Visualize what loss means - the errors
fig, ax = plt.subplots(figsize=(10, 6))

# Use the "good" line
slope, intercept = 10, 50
predictions = predict(sizes, slope, intercept)

# Data points
ax.scatter(sizes, prices, alpha=0.7, s=60, label='Actual data', zorder=3)

# Line
x_line = np.linspace(8, 32, 100)
y_line = slope * x_line + intercept
ax.plot(x_line, y_line, 'r-', linewidth=2, label='Prediction line')

# Draw error lines
for i in range(len(sizes)):
    ax.plot([sizes[i], sizes[i]], [prices[i], predictions[i]], 
            'g-', alpha=0.5, linewidth=1)

ax.set_xlabel('Size (hundreds of sq ft)')
ax.set_ylabel('Price (thousands of $)')
ax.set_title('The Loss = Sum of Squared Green Lines')
ax.legend()
plt.show()

print("\n📏 The green lines show the ERRORS (prediction - actual)")
print("   MSE squares these and averages them.")

## 4. Gradient Descent: "Walk Downhill"

Now for the **magic**: How do we find the best weight and bias?

Imagine the loss as a landscape (higher = worse). We want to find the lowest point.

**Gradient Descent**: Start somewhere, then repeatedly:
1. Look which way is "downhill" (calculate the gradient)
2. Take a small step in that direction
3. Repeat until you reach the bottom

It's like being blindfolded on a hill and finding the valley by feeling which way is down!

In [ ]:
# Let's visualize the loss landscape for different weights
# (keeping bias fixed for simplicity)

weights_to_try = np.linspace(0, 20, 100)
losses = []

for w in weights_to_try:
    preds = predict(sizes, w, 50)  # fixed bias = 50
    loss = mse_loss(preds, prices)
    losses.append(loss)

plt.figure(figsize=(10, 5))
plt.plot(weights_to_try, losses, 'b-', linewidth=2)
plt.xlabel('Weight (slope)')
plt.ylabel('Loss (MSE)')
plt.title('Loss Landscape - Finding the Minimum')

# Mark the minimum
min_idx = np.argmin(losses)
plt.scatter([weights_to_try[min_idx]], [losses[min_idx]], 
            c='red', s=200, zorder=5, label=f'Minimum at w={weights_to_try[min_idx]:.1f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"\n🎯 The best weight is around {weights_to_try[min_idx]:.1f}")
print(f"   (True weight we used to generate data: {true_slope})")

In [ ]:
# Now let's implement gradient descent!

def compute_gradients(X, y, weight, bias):
    """
    Compute gradients of MSE loss with respect to weight and bias.
    
    The gradient tells us: "which direction increases the loss?"
    We'll move in the OPPOSITE direction to decrease loss.
    """
    n = len(X)
    predictions = predict(X, weight, bias)
    errors = predictions - y
    
    # Gradient of MSE with respect to weight
    # d(MSE)/d(weight) = (2/n) * sum(errors * X)
    d_weight = (2/n) * np.sum(errors * X)
    
    # Gradient of MSE with respect to bias
    # d(MSE)/d(bias) = (2/n) * sum(errors)
    d_bias = (2/n) * np.sum(errors)
    
    return d_weight, d_bias

def gradient_descent(X, y, learning_rate=0.001, n_iterations=1000):
    """
    Find the best weight and bias using gradient descent.
    
    learning_rate: How big of steps to take (too big = overshoot, too small = slow)
    """
    # Start with random values
    weight = np.random.randn()
    bias = np.random.randn()
    
    # Track history for visualization
    history = {'weight': [weight], 'bias': [bias], 'loss': []}
    
    for i in range(n_iterations):
        # Calculate current loss
        predictions = predict(X, weight, bias)
        loss = mse_loss(predictions, y)
        history['loss'].append(loss)
        
        # Calculate gradients
        d_weight, d_bias = compute_gradients(X, y, weight, bias)
        
        # Update parameters (move opposite to gradient)
        weight = weight - learning_rate * d_weight
        bias = bias - learning_rate * d_bias
        
        # Save history
        history['weight'].append(weight)
        history['bias'].append(bias)
        
        # Print progress
        if i % 200 == 0:
            print(f"Iteration {i:4d}: Loss = {loss:.2f}, Weight = {weight:.2f}, Bias = {bias:.2f}")
    
    return weight, bias, history

# Run gradient descent!
print("Training...\n")
learned_weight, learned_bias, history = gradient_descent(sizes, prices, learning_rate=0.001, n_iterations=1000)

print(f"\n✅ Final: Weight = {learned_weight:.2f}, Bias = {learned_bias:.2f}")
print(f"   True:  Weight = {true_slope}, Bias = {true_intercept}")

In [ ]:
# Visualize the learning process
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss over time
axes[0].plot(history['loss'], 'b-', linewidth=1)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Loss (MSE)')
axes[0].set_title('Loss Decreasing Over Time')
axes[0].set_yscale('log')  # Log scale to see the decrease better

# Final fit
axes[1].scatter(sizes, prices, alpha=0.7, s=60, label='Data')
x_line = np.linspace(8, 32, 100)

# True line (dashed)
y_true = true_slope * x_line + true_intercept
axes[1].plot(x_line, y_true, 'g--', linewidth=2, label=f'True: y = {true_slope}x + {true_intercept}')

# Learned line (solid)
y_learned = learned_weight * x_line + learned_bias
axes[1].plot(x_line, y_learned, 'r-', linewidth=2, 
             label=f'Learned: y = {learned_weight:.1f}x + {learned_bias:.1f}')

axes[1].set_xlabel('Size')
axes[1].set_ylabel('Price')
axes[1].set_title('Final Fit')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n🎉 The model LEARNED the relationship from data!")

## 5. Making Predictions

Now we can predict prices for houses we've never seen!

In [ ]:
# Predict prices for new houses
new_sizes = np.array([15, 20, 25, 35])  # New house sizes
predicted_prices = predict(new_sizes, learned_weight, learned_bias)

print("Predictions for new houses:")
print("=" * 50)
for size, price in zip(new_sizes, predicted_prices):
    print(f"Size: {size*100:,} sq ft → Predicted price: ${price*1000:,.0f}")

In [ ]:
# Visualize predictions
plt.figure(figsize=(10, 6))

# Training data
plt.scatter(sizes, prices, alpha=0.5, s=40, label='Training data')

# Fitted line
x_line = np.linspace(5, 40, 100)
y_line = learned_weight * x_line + learned_bias
plt.plot(x_line, y_line, 'r-', linewidth=2, label='Learned model')

# New predictions
plt.scatter(new_sizes, predicted_prices, c='green', s=200, marker='*', 
            edgecolors='black', linewidth=1, label='New predictions', zorder=5)

# Annotate predictions
for size, price in zip(new_sizes, predicted_prices):
    plt.annotate(f'${price*1000:,.0f}', (size, price), 
                 textcoords='offset points', xytext=(0, 15), ha='center')

plt.xlabel('Size (hundreds of sq ft)')
plt.ylabel('Price (thousands of $)')
plt.title('Making Predictions with Our Trained Model')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 6. The Learning Rate: A Crucial Hyperparameter

The **learning rate** controls how big our steps are during gradient descent.

- **Too small**: Very slow learning, might take forever
- **Too large**: Overshoots the minimum, might never converge
- **Just right**: Quick and stable convergence

In [ ]:
# Compare different learning rates
learning_rates = [0.0001, 0.001, 0.01, 0.1]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, lr in enumerate(learning_rates):
    # Set same starting point
    np.random.seed(42)
    
    _, _, hist = gradient_descent(sizes, prices, learning_rate=lr, n_iterations=500)
    
    ax = axes[idx]
    
    # Handle potential infinity/overflow for large learning rates
    losses = np.array(hist['loss'])
    losses = np.clip(losses, 0, 1e10)  # Clip extremely large values
    
    ax.plot(losses, linewidth=1)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title(f'Learning Rate = {lr}')
    
    if max(losses) > 1000:
        ax.set_yscale('log')
        ax.text(0.5, 0.5, 'DIVERGING!', transform=ax.transAxes, 
                fontsize=20, color='red', ha='center', alpha=0.7)
    elif idx == 0:
        ax.text(0.5, 0.5, 'Too slow!', transform=ax.transAxes, 
                fontsize=16, color='orange', ha='center', alpha=0.7)

plt.tight_layout()
plt.show()

print("\n💡 Learning rate 0.001 or 0.01 works well for this problem!")

## 7. Connecting to NLP

How does this relate to text and language? Linear regression is the **building block** for:

- **Sentiment scoring**: Given word counts (features), predict a sentiment score
- **Text similarity**: Given embeddings, predict similarity scores
- **Every neural network layer**: It's basically a fancy linear regression + activation!

Let's see a simple NLP example:

In [ ]:
# Simple sentiment example
# Feature: count of positive words vs negative words

# Simulated data: (positive_word_count - negative_word_count) → sentiment score
np.random.seed(42)

n_texts = 30
word_difference = np.random.uniform(-5, 5, n_texts)  # pos words - neg words
sentiment_score = 0.5 + 0.1 * word_difference + np.random.normal(0, 0.05, n_texts)  # 0-1 scale
sentiment_score = np.clip(sentiment_score, 0, 1)

# Train our linear regression
np.random.seed(42)
w, b, _ = gradient_descent(word_difference, sentiment_score, learning_rate=0.01, n_iterations=500)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(word_difference, sentiment_score, s=60, alpha=0.7)

x_line = np.linspace(-6, 6, 100)
y_line = w * x_line + b
plt.plot(x_line, y_line, 'r-', linewidth=2, label=f'Model: score = {w:.2f} × diff + {b:.2f}')

plt.xlabel('(Positive Words) - (Negative Words)')
plt.ylabel('Sentiment Score (0=negative, 1=positive)')
plt.title('Simple NLP: Predicting Sentiment from Word Counts')
plt.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Neutral')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\n📝 Example predictions:")
examples = [(-3, 'Text with more negative words'), (0, 'Neutral text'), (3, 'Text with more positive words')]
for diff, desc in examples:
    pred = w * diff + b
    sentiment = 'Positive' if pred > 0.5 else 'Negative'
    print(f"  {desc}: score = {pred:.2f} ({sentiment})")

## ⚠️ What Can Go Wrong: Learning Rate Explosion

The learning rate is the most common source of training failures.
Too large → weights explode → loss becomes NaN. Let's see it happen.

In [ ]:
# --- What happens with a HUGE learning rate? ---
np.random.seed(42)

# Try lr = 1.0 (way too big for this data)
w_explode = np.random.randn()
b_explode = np.random.randn()
lr_too_big = 1.0

print("Training with lr = 1.0 (too big!)")
print("=" * 50)

explode_losses = []
for i in range(20):
    preds = w_explode * sizes + b_explode
    loss = np.mean((preds - prices) ** 2)
    explode_losses.append(min(loss, 1e15))  # Cap for plotting
    
    # Gradients
    errors = preds - prices
    d_w = (2/len(sizes)) * np.sum(errors * sizes)
    d_b = (2/len(sizes)) * np.sum(errors)
    
    w_explode -= lr_too_big * d_w
    b_explode -= lr_too_big * d_b
    
    if i < 5 or np.isnan(loss) or np.isinf(loss):
        print(f"  Step {i}: loss = {loss:.2e}, w = {w_explode:.2e}")
    if np.isnan(loss) or np.isinf(loss):
        print("  💥 EXPLODED! Loss is NaN/Inf!")
        break

# Now fix it with a smaller lr
print("\n--- Fix: reduce learning rate to 0.001 ---")
np.random.seed(42)
w_fix, b_fix, hist_fix = gradient_descent(sizes, prices, learning_rate=0.001, n_iterations=500)
print(f"\n✅ Fixed! Final weight={w_fix:.2f}, bias={b_fix:.2f}")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(explode_losses, 'r-', linewidth=2)
axes[0].set_title('lr = 1.0 (EXPLODING! 💥)')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_yscale('log')

axes[1].plot(hist_fix['loss'], 'g-', linewidth=2)
axes[1].set_title('lr = 0.001 (stable ✅)')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Loss')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print("\n🔑 Rule of thumb: if loss is NaN → first try a SMALLER learning rate!")

## 📝 Check Your Understanding

1. What does the loss function measure?
2. Why do we square the errors in MSE?
3. What does the gradient tell us?
4. What happens if the learning rate is too high? Too low?
5. In `y = wx + b`, what do `w` and `b` represent?

In [ ]:
# --- Exercise 1: Make a Prediction ---
# Given weight=10 and bias=50, predict the price for a house of size 20.
# Use the formula: prediction = weight * size + bias

# YOUR CODE HERE:
predicted_price = None  # Replace with your prediction

# --- Check ---
assert predicted_price is not None, "Replace None with your answer!"
assert predicted_price == 250, f"Expected 250 (10×20 + 50), got {predicted_price}"
print("Exercise 1 passed! ✓")

# --- Exercise 2: Compute MSE ---
# Calculate the Mean Squared Error for these predictions vs actuals.
ex_predictions = np.array([100, 200, 300])
ex_actuals = np.array([110, 190, 310])

# YOUR CODE HERE:
mse = None  # Compute MSE: mean of (predictions - actuals)^2

# --- Check ---
assert mse is not None, "Replace None with your answer!"
expected_mse = np.mean((ex_predictions - ex_actuals) ** 2)
assert abs(mse - expected_mse) < 0.01, f"Expected {expected_mse:.2f}, got {mse:.2f}"
print(f"Exercise 2 passed! ✓  (MSE = {mse:.2f})")

# --- Quick Check: Gradient Direction ---
# If increasing the weight makes the loss INCREASE, what should we do?
# a) Decrease the weight (move opposite to gradient)
# b) Increase the weight more
# c) Keep the weight the same
# d) Set the weight to zero

your_answer = None  # Put 'a', 'b', 'c', or 'd'

# --- Check ---
assert your_answer is not None, "Pick an answer!"
assert your_answer == 'a', "The gradient points uphill — we want to go DOWNHILL, so we move in the opposite direction!"
print("Exercise 3 passed! ✓")

print("\n🎉 All exercises passed!")

## 🎯 Summary

You learned:
- **Linear regression** finds the best line through data
- **Loss function** measures how wrong our predictions are
- **Gradient descent** finds the best parameters by "walking downhill"
- **Learning rate** controls step size (crucial hyperparameter!)
- These concepts are the foundation of ALL neural networks!

### The Big Picture

```
Linear Regression:
y = weight × x + bias

Neural Network Layer:
output = activation(weights × input + biases)
```

Neural networks are just many linear regressions stacked together with non-linear activations!

**Next up**: Classification - making yes/no decisions! →